In [1]:
# C1 - Imports et chargement des résultats

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df_v1 = pd.read_csv("results_v1.csv")
df_v2 = pd.read_csv("results_v2.csv")
df_v3 = pd.read_csv("results_v3.csv")

df_all = pd.concat([df_v1, df_v2, df_v3], ignore_index=True)

# Moyennes par variante
df_moyennes = (
    df_all
    .groupby("Variante")[["WMAE","MAE","RMSE","R2"]]
    .mean()
    .round(2)
    .reset_index()
)

df_moyennes["WMAE/RMSE"] = (df_moyennes["WMAE"] / df_moyennes["RMSE"]).round(4)

print("=== Comparaison des moyennes par variante ===")
display(df_moyennes)

=== Comparaison des moyennes par variante ===


,Variante,WMAE,MAE,RMSE,R2,WMAE/RMSE
0,V1 - Brut + Target Encoding,2327.61,1851.45,5242.34,0.94,0.4440
1,V2 — Log + sans négatifs + Target Encoding,2282.22,1803.58,5223.44,0.94,0.4369
2,V3 — Imputation 0 + flag + Target Encoding,2327.02,1851.66,5245.46,0.94,0.4436



- V2 domine sur tous les critères : WMAE, MAE, RMSE et ratio WMAE/RMSE
- V3 quasi identique à V1 - le flag was_negative n'apporte pas d'amélioration significative
- R² identique pour les 3 variantes - même pouvoir explicatif global
- Variante retenue : V2 - Log + sans négatifs + Target Encoding

In [6]:
# C2 - Visualisation comparative des moyennes

variantes_short = {
    "V1 - Brut + Target Encoding"                    : "V1",
    "V2 — Log + sans négatifs + Target Encoding"     : "V2",
    "V3 — Imputation 0 + flag + Target Encoding"     : "V3"
}
df_moyennes["Variante_short"] = df_moyennes["Variante"].map(variantes_short)

colors = ["#636EFA", "#00CC96", "#EF553B"]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("WMAE moyen", "RMSE moyen", "Ratio WMAE/RMSE"),
    horizontal_spacing=0.12
)

for i, (col, row, title) in enumerate([
    ("WMAE",      1, "WMAE moyen ($)"),
    ("RMSE",      1, "RMSE moyen ($)"),
    ("WMAE/RMSE", 1, "Ratio WMAE/RMSE"),
]):
    for j, (_, row_data) in enumerate(df_moyennes.iterrows()):
        fig.add_trace(go.Bar(
            name=row_data["Variante_short"],
            x=[row_data["Variante_short"]],
            y=[row_data[col]],
            marker_color=colors[j],
            showlegend=(i == 0),
            hovertemplate=f"{col}: %{{y:,.2f}}<extra>{row_data['Variante_short']}</extra>"
        ), row=1, col=i+1)

fig.update_layout(
    title_text="Comparaison RF - V1 vs V2 vs V3 - Moyennes",
    title_font_size=16,
    barmode="group",
    height=420,
    template="plotly_white",
    legend=dict(orientation="h", y=-0.2, x=0.5, xanchor="center")
)
fig.show()

# C3 - Verdict final

verdict = """
### Verdict - Selection du modele Random Forest

#### Modele retenu : V2 - Log + sans negatifs + Target Encoding

| Critere         | V1 - Brut | V2 - Log | V3 - Flag |
|-----------------|-----------|----------|-----------|
| WMAE moyen      | 2 328 $   | 2 282 $  | 2 327 $   |
| MAE moyen       | 1 851 $   | 1 804 $  | 1 852 $   |
| RMSE moyen      | 5 242 $   | 5 223 $  | 5 245 $   |
| R2 moyen        | 0.94      | 0.94     | 0.94      |
| Ratio WMAE/RMSE | 0.444     | 0.437    | 0.444     |

#### Justification
- V2 domine sur tous les criteres - WMAE, MAE, RMSE et ratio WMAE/RMSE
- La transformation log stabilise les predictions sur les ventes elevees
- V3 n'apporte rien par rapport a V1 - le flag was_negative est insuffisant
- Ecarts modestes (+2%) - le vrai gain attendu en C6 tuning et Bloc D



In [5]:
# Comparaison V1 vs V2 vs V3 — WMAE par fold avec décalage visuel

folds = ["Fold 1", "Fold 2", "Fold 3", "Fold 4", "Fold 5"]
folds_x_v1 = [0.9, 1.9, 2.9, 3.9, 4.9]
folds_x_v2 = [1.0, 2.0, 3.0, 4.0, 5.0]
folds_x_v3 = [1.1, 2.1, 3.1, 4.1, 5.1]

wmae_v1 = [4288, 1525, 2568, 1893, 1364]
wmae_v2 = [4313, 1491, 2445, 1845, 1318]
wmae_v3 = [4286, 1527, 2566, 1893, 1363]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=folds_x_v1, y=wmae_v1,
    mode="lines+markers+text",
    name="V1 - Brut",
    line=dict(color="#636EFA", width=2, dash="dot"),
    marker=dict(size=10),
    text=[f"{v:,.0f}$" for v in wmae_v1],
    textposition="top left"
))

fig.add_trace(go.Scatter(
    x=folds_x_v2, y=wmae_v2,
    mode="lines+markers+text",
    name="V2 - Log (retenue)",
    line=dict(color="#00CC96", width=3),
    marker=dict(size=12, symbol="star"),
    text=[f"{v:,.0f}$" for v in wmae_v2],
    textposition="bottom center"
))

fig.add_trace(go.Scatter(
    x=folds_x_v3, y=wmae_v3,
    mode="lines+markers+text",
    name="V3 - Flag",
    line=dict(color="#EF553B", width=2, dash="dash"),
    marker=dict(size=10),
    text=[f"{v:,.0f}$" for v in wmae_v3],
    textposition="top right"
))

fig.update_layout(
    title="Erreur de prediction (WMAE) par periode — plus bas = meilleur",
    title_font_size=15,
    xaxis=dict(
        tickvals=[1, 2, 3, 4, 5],
        ticktext=folds,
        title="Periode de validation"
    ),
    yaxis_title="Erreur moyenne ponderee ($)",
    template="plotly_white",
    height=480,
    legend=dict(orientation="h", y=-0.15, x=0.5, xanchor="center")
)

fig.show()